In [ ]:
PROJECT_REF = "main"


def _install_project(project_ref: str):
    import urllib.parse
    import urllib.request

    from google.colab import userdata

    github_token = userdata.get("GITHUB_TOKEN_JLENS_REAS")
    if not github_token:
        raise RuntimeError(
            "Required Colab secret GITHUB_TOKEN_JLENS_REAS is unavailable"
        )

    query = urllib.parse.urlencode({"ref": project_ref})
    bootstrap_url = (
        "https://api.github.com/repos/noamdwc/jlens-reasoning/"
        "contents/scripts/colab_bootstrap.py?" + query
    )
    request = urllib.request.Request(
        bootstrap_url,
        headers={
            "Authorization": f"Bearer {github_token}",
            "Accept": "application/vnd.github.raw+json",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    try:
        with urllib.request.urlopen(request) as response:
            bootstrap_source = response.read().decode("utf-8")
    except Exception:
        raise RuntimeError("Unable to load the Colab bootstrap") from None

    namespace = {}
    exec(
        compile(
            bootstrap_source,
            "scripts/colab_bootstrap.py",
            "exec",
        ),
        namespace,
    )
    return namespace["bootstrap"](
        project_ref=project_ref,
        github_token=github_token,
    )


PROJECT_DIR = _install_project(PROJECT_REF)
del _install_project

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
context

In [ ]:
import importlib.metadata
import subprocess

import jlens
import torch
import transformers

from jlens_reasoning.experiments.readout_sanity import (
    LENS_FILE,
    LENS_REPO,
    LENS_REVISION,
    MODEL_NAME,
    READOUT_CASES,
    concept_token_variants,
    run_readout_sanity,
    validate_model_lens,
    write_results,
)

causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(causal_lm, tokenizer)
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO,
    filename=LENS_FILE,
    revision=LENS_REVISION,
)
validate_model_lens(model, lens)
model, lens

In [ ]:
@torch.inference_mode()
def forward_next_token(input_ids):
    return causal_lm(input_ids=input_ids, use_cache=False).logits[0, -1]


result = run_readout_sanity(
    model=model,
    lens=lens,
    tokenizer=tokenizer,
    unembedding_weight=causal_lm.get_output_embeddings().weight,
    forward_next_token=forward_next_token,
)
result["provenance"] = {
    "project_commit": subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "jlens": importlib.metadata.version("jlens"),
}

run_dir = context.runs_dir / "jlens-readout-sanity"
result_path = run_dir / "result.json"
write_results(result_path, result)

for case in result["cases"]:
    summary = case["summary"]["jacobian_lens"]
    print(
        case["key"],
        f"baseline={case['baseline']['top1_token']!r}",
        f"read_rank={summary['best_rank']}",
        f"read_layer={summary['layer']}",
        f"read_position={summary['position']}",
    )
for swap in result["swaps"]:
    print(
        swap["key"],
        f"clean_target_rank={swap['clean']['target_rank']}",
        f"alpha1_rank={swap['interventions']['1.0']['target_rank']}",
        f"alpha2_rank={swap['interventions']['2.0']['target_rank']}",
        f"improved={swap['improved']}",
        f"target_top1={swap['target_top1']}",
    )
print("intervention_strengths", result["intervention_strengths"])

controls = result["controls"]
identity = controls["identity"]
print(
    "identity_control",
    f"passed={identity['passed']}",
    (f"max_abs_logit_difference={identity['maximum_absolute_logit_difference']}"),
)
matched_random = controls["matched_random_vector"]
print(
    "matched_random_vector_control",
    f"passed={matched_random['passed']}",
    f"real_mean={matched_random['real_mean_log_rank_gain']}",
    f"percentile_95={matched_random['percentile_95_threshold']}",
)
wrong = controls["wrong_concept"]
print(
    "wrong_concept_control",
    f"passed={wrong['passed']}",
    f"matched_mean={wrong['matched_mean_log_rank_gain']}",
    f"mismatched_mean={wrong['mismatched_mean_log_rank_gain']}",
    f"matched_case_wins={wrong['matched_winning_case_count']}/5",
)
random_target = controls["random_target"]
print(
    "random_target_control",
    f"passed={random_target['passed']}",
    f"real_mean={random_target['real_mean_log_rank_gain']}",
    f"percentile_95={random_target['percentile_95_threshold']}",
)
print("overall_controls", f"passed={controls['passed']}")
print(f"Saved: {result_path}")

In [ ]:
from IPython.display import display
from jlens.vis import build_page, compute_slice, notebook_iframe

for case in (READOUT_CASES[0], READOUT_CASES[1]):
    pinned = {
        variant.token_id
        for variant in concept_token_variants(tokenizer, case.target_concepts)
    }
    slice_data = compute_slice(
        model,
        lens,
        case.prompt,
        top_n=25,
        pinned_token_ids=pinned,
        mask_display=True,
    )
    page, _, _ = build_page(
        slice_data,
        case.prompt,
        title=f"J-Lens read-and-change sanity: {case.key}",
        description="Paper-aligned read-and-change open-model sanity check.",
        mode="embed",
    )
    html_path = run_dir / f"{case.key}.html"
    html_path.write_text(page, encoding="utf-8")
    print(f"Saved: {html_path}")
    display(notebook_iframe(page))

In [ ]:
if not result["passed"]:
    raise RuntimeError(
        "Read-and-change sanity checks failed: " + "; ".join(result["failures"])
    )

print("All J-Lens read-and-change sanity checks passed.")